# 07. 공통 기준 테이블 만들기

전체 345,390행을 보존하고 제외 표시·파생컬럼·근거·임시 LABEL을 붙임. CP 단독 조건이나 LOT 전체 수준 차이로 1을 만들지 않음.
`processed/cal01_base_v1.csv` 하나가 공통 기준임. 분류·회귀 폴더에는 이 표를 읽는 정의만 둠.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/README.md').is_file() and (p / 'notebooks').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from namelesspark_pipeline import load_sources, build_lot_context, build_maintenance_context, apply_user_exclusions
from namelesspark_weak_labels import build_weak_labels
from namelesspark_datasets import build_master, export_master, classification_inputs

PROCESSED = ROOT / 'data/processed'
REVIEW = PROCESSED / 'review/v1'
SIGNALS = PROCESSED / 'analysis/signals/v1'
FEATURE_FILE = PROCESSED / 'features/v1/06_features_1min.csv'
CANDIDATE_FILE = SIGNALS / '05_all_sensor_sustained_candidates.csv'
ROW_LINK_FILE = REVIEW / '05_domain_row_index.csv'
for path in [FEATURE_FILE, CANDIDATE_FILE, ROW_LINK_FILE]:
    if not path.is_file():
        raise FileNotFoundError(f'05·06을 먼저 실행해야 함: {path}')
sources = load_sources(ROOT)
marked, exclusions = apply_user_exclusions(sources['temp'])
lot_context = build_lot_context(sources['temp'], sources['event'])
marked['GRADE'] = marked.LOT_NO.map(lot_context.set_index('LOT_NO').GRADE)
maintenance, part_details = build_maintenance_context(sources['maint'], sources['partchg'], sources['part'])
features = pd.read_csv(FEATURE_FILE, encoding='utf-8-sig', parse_dates=['MEAS_DT'])
candidates = pd.read_csv(CANDIDATE_FILE, encoding='utf-8-sig', parse_dates=['START', 'END'])
row_links = pd.read_csv(ROW_LINK_FILE, encoding='utf-8-sig', parse_dates=['MEAS_DT'])

## 임시 0·1 정의

1은 강한 센서 지속 변화와 적격 정비 기록의 시간 연관이 함께 있는 행임. 0은 이번 규칙 미검출이며 확정 정상은 아님. 사용자 제외 행은 LABEL을 비워두고 학습에서 뺌.
공정 허용치·고장 원인을 확정한 정답이 아니며, 문턱과 시간 범위는 실험 가정임. 자세한 조건은 `metadata/v1/dataset.json`에 남김.

In [ ]:
row_labels, interval_decisions, label_policy = build_weak_labels(marked, candidates, sources, maintenance)
master, input_columns = build_master(sources, features, row_labels, row_links)
REVIEW.mkdir(parents=True, exist_ok=True)
source_files = [ROOT/'data'/name for name in [
    'T-CR1-CAL01_온도.csv', 'G-02_조업이벤트.csv', 'G-01_정기수리캘린더.csv',
    'G-01_설비마스터.csv', 'G-01_부품마스터.csv', 'G-02_정비이력.csv', 'G-02_부품교체이력.csv',
]] + [FEATURE_FILE, CANDIDATE_FILE, ROW_LINK_FILE,
      ROOT/'src/namelesspark_weak_labels.py', ROOT/'src/namelesspark_datasets.py',
      ROOT/'src/namelesspark_pipeline.py', ROOT/'src/namelesspark_temporal_review.py']
master_path, manifest = export_master(ROOT, master, input_columns, label_policy, source_files)
interval_decisions.to_csv(REVIEW / 'weak_label_interval_decisions.csv', index=False, encoding='utf-8-sig')
print('공통 기준:', master_path)
print('전체:', len(master), '/ 임시 0:', manifest['label_0'], '/ 임시 1:', manifest['label_1'])
print('사용자 제외:', manifest['excluded_rows'], '/ 입력 완전한 학습 후보:', manifest['model_eligible_rows'])
print('임시 1의 센서별 기여:', label_policy['positive_contribution_by_sensor'])

## 모델 입력 구분

Z-score·IsolationForest·RandomForestClassifier는 같은 X와 평가 행을 사용함. LABEL은 평가 정답 또는 분류 목표값이고 X에는 넣지 않음.
회귀는 같은 기준표에서 목표 센서·미래 시차를 정한 뒤 생성함. 아직 목표가 미정이라 회귀 정답을 임의로 만들지 않음. 분할·9:1 표본 추출·학습은 여기서 하지 않음.

In [ ]:
X, y, row_keys = classification_inputs(master, input_columns)
assert X.select_dtypes(include='number').shape[1] == X.shape[1]
assert X.notna().all().all() and set(y.unique()).issubset({0, 1})
assert not set(['LABEL', 'RULE', 'EVIDENCE_WO', 'MEAS_DT', 'LOT_NO', 'EXCLUDED']).intersection(X.columns)
print('X:', X.shape, '/ y:', y.value_counts().to_dict())
print('분류 정의:', PROCESSED / 'classification/v1/dataset.json')
print('회귀 정의:', PROCESSED / 'regression/v1/dataset.json')